# Rent model v1 - your teammate's submission (audit me)

A teammate trained this overnight and wants to ship it:

> *"log-Rent R^2 = 0.83, average error under 0.3, basically done. Pushing to prod."*

**Your job (full instructions in the homework brief):** this notebook hides
**six** methodology bugs from L04-L07. **Find** them (Part 1 audit table),
**fix** them (Part 2), and report the **honest** number with a model card (Part 3).

> [!WARNING]
> Most of these bugs do **not** change the headline number - broken methodology
> can still print a great score. Don't hunt for a value that looks off; read the
> code against the L04-L07 rules. (The honest, sealed-test score does come out a
> little lower, but that is mostly one bug, not all six.)

## Teammate's notes

Standard regression on the Kaggle house-rent data. I log-transformed `Rent`
(it's skewed), engineered a smart locality feature, regularized with Ridge,
tuned `alpha`, and checked the metrics. Clean run, great score.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

SEED = 509
np.random.seed(SEED)

### Load the data + a clever feature

Average rent per locality is obviously predictive, so I add it as a feature.

In [2]:
df = pd.read_csv("../01_regression_intro/data/House_Rent_Dataset.csv")

# "locality_mean_rent": the average rent in each Area Locality. Strong signal!
df["locality_mean_rent"] = df.groupby("Area Locality")["Rent"].transform("mean")

print("shape:", df.shape)
df[["Area Locality", "Rent", "locality_mean_rent"]].head()

shape: (4746, 13)


,Area Locality,Rent,locality_mean_rent
0,Bandel,10000,8250.0
1,"Phool Bagan, Kankurgachi",20000,11750.0
2,Salt Lake City Sector 2,17000,23187.5
3,Dumdum Park,10000,16000.0
4,South Dum Dum,7500,7500.0


### Encode and build the design matrix

Numeric features + the locality feature, plus one-hot encoded categoricals.
Target is `log(1 + Rent)`.

In [3]:
num = ["BHK", "Size", "Bathroom", "locality_mean_rent"]
cat = ["City", "Furnishing Status", "Area Type"]

# one-hot encode the categoricals
ohe = OneHotEncoder(handle_unknown="ignore", drop="first", sparse_output=False)
X_cat = ohe.fit_transform(df[cat])

X = np.hstack([df[num].values, X_cat])
y = np.log1p(df["Rent"].values)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED)
print("train:", X_train.shape, " test:", X_test.shape)

train: (3796, 13)  test: (950, 13)


### Tune `alpha`

Sweep a range of penalties and keep whichever scores best.

In [4]:
best_alpha, best_score = None, -np.inf
for alpha in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
    m = Ridge(alpha=alpha).fit(X_train, y_train)
    score = r2_score(y_test, m.predict(X_test))   # score each alpha on the test set
    if score > best_score:
        best_alpha, best_score = alpha, score

print(f"best alpha = {best_alpha}   (test R2 = {best_score:.3f})")

best alpha = 1.0   (test R2 = 0.833)


### Fit the regularized model and look at coefficients

Ridge with the best alpha. Quick sanity check on the coefficients.

In [5]:
final = Ridge(alpha=best_alpha).fit(X_train, y_train)

for name, coef in zip(num, final.coef_[:len(num)]):
    print(f"  {name:<20} {coef: .5f}")
print("\nCoefficients look reasonable. Going with this model.")

  BHK                   0.22657
  Size                  0.00038
  Bathroom              0.11758
  locality_mean_rent    0.00000

Coefficients look reasonable. Going with this model.


### Results

Final metrics on the held-out test set.

In [6]:
p = final.predict(X_test)
print(f"R2   = {r2_score(y_test, p):.3f}")
print(f"RMSE = {mean_squared_error(y_test, p) ** 0.5:.3f}")
print(f"MAE  = {mean_absolute_error(y_test, p):.3f}")

# cross-check R2 on just the normal-priced flats (drop the few luxury ones)
mask = y_test < np.log1p(100_000)
print(f"R2 on normal flats only = {r2_score(y_test[mask], p[mask]):.3f}")

# report the CV score as the final number
cv = cross_val_score(Ridge(alpha=best_alpha), X_train, y_train,
                     cv=KFold(5, shuffle=True, random_state=SEED), scoring="r2")
print(f"\nFINAL reported score (CV R2) = {cv.mean():.3f}")

R2   = 0.833
RMSE = 0.372
MAE  = 0.278
R2 on normal flats only = 0.759

FINAL reported score (CV R2) = 0.828


## Conclusion

log-Rent **R^2 ~ 0.83**, average error **under 0.3**, and R^2 holds up on the
normal-priced flats too. The locality feature really paid off. Model's basically
done - shipping it. 🚀